In [20]:
import numpy as np
import pandas as pd
import re
from word2number import w2n

# Functions
def fix_quantity(x) -> int:
    if isinstance(x, str):
        x_clean = x.strip().lower()
        try:
            # try converting word numbers
            return w2n.word_to_num(x_clean)
        except:
            try:
                # try converting numeric strings
                return int(x_clean)
            except:
                return x # return as is if failed conversion
    return x

def try_parse_date(date_str):
    if pd.isna(date_str):
        return pd.NaT
    
    # Remove spaces
    date_str = str(date_str).strip()

    # ISO format: YYYY-MM-DD or YYYY/MM/DD
    iso_pattern = r"^\d{4}[-/]\d{2}[-/]\d{2}$"

    try:
        if re.match(iso_pattern, date_str):
            # Parse normally (ISO)
            return pd.to_datetime(date_str, errors='coerce')
        else:
            # Try day-first formats (like 15-02-2020)
            return pd.to_datetime(date_str, errors='coerce', dayfirst=True)
    except:
        return pd.NaT
        
def convert_rating(rating) -> int | None:
    rating=str(rating).strip()
    if rating.isdigit():
        return int(rating)

    stars={'⭐', '★'}
    star_count=sum(1 for c in rating if c in stars)

    return star_count

# Setup
df=pd.read_csv('flower_shop_orders.csv')
df=df.rename(columns={'PricePerStem': 'PricePerStem ($)'})

# Name
df['CustomerName']=df['CustomerName'].str.title()

# Email
email_pattern = r"^(?!.*\.\.)[\w\.-]+@[\w-]+(\.[\w-]+)+$"
df['Email']=df['Email'].apply(lambda x: x if re.match(email_pattern, str(x)) else np.nan)
df['EmailMissing'] = df['Email'].isnull()

# Flower Type
flower_type_map={
    'roses': 'Rose',
    'rose': 'Rose',
    'tulips': 'Tulip',
    'lilies': 'Lily',
    'sun flower': 'Sunflower',
    'sunflowers': 'Sunflower',
    'orchid': 'Orchid',
    'orchids': 'Orchid'
}
df['FlowerType']=df['FlowerType'].str.lower()
df['FlowerType']=df['FlowerType'].map(flower_type_map)
df=pd.get_dummies(df, columns=['FlowerType'], drop_first=True)

# Quantity
df['Quantity']=df['Quantity'].apply(fix_quantity)
df['Quantity']=df['Quantity'].apply(lambda x: max(x, 0))

# PricePerStem
df['PricePerStem ($)']=df['PricePerStem ($)'].str.replace(',', '.')
df['PricePerStem ($)']=df['PricePerStem ($)'].str.replace('$', '')

# OrderDate
df['OrderDate']=df['OrderDate'].apply(try_parse_date)
df['OrderDate'] = df['OrderDate'].fillna(df['OrderDate'].median())

# DeliveryDate
df['DeliveryDate']=df['DeliveryDate'].apply(try_parse_date)
df['DeliveryDate'] = df['DeliveryDate'].fillna(df['DeliveryDate'].median())

df = df[df['DeliveryDate'] >= df['OrderDate']]

# Rating
df['Rating']=df['Rating'].str.replace('stars', '')
df['Rating']=df['Rating'].apply(convert_rating)

# DeliveryStatus
df['DeliveryStatus']=df['DeliveryStatus'].str.lower()
df['DeliveryStatus'] = df['DeliveryStatus'].replace({'deliverd': 'delivered'})
delivery_status_map={
    'pending': 0,
    'in transit': 1,
    'delivered': 2
}
df['DeliveryStatus'] = df['DeliveryStatus'].map(delivery_status_map)

# PaymentStatus
df['PaymentStatus']=df['PaymentStatus'].str.lower()
payment_status_map={
    'unpaid': 0,
    'payed': 1,
    'paid': 1
}
df['PaymentStatus']=df['PaymentStatus'].map(payment_status_map)

# Others
df=df.drop_duplicates()
df=df.drop(columns=['OrderID', 'Review'])

# Output
# Force Display
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
df
# df.isna().sum()
# for col in df.columns:
#     print(f"\n{col} ➤ Unique values:")
#     print(df[col].dropna().unique())

CustomerName            0
Email                   5
Quantity                0
PricePerStem ($)        0
OrderDate               0
DeliveryDate            0
Rating                  0
DeliveryStatus          0
PaymentStatus           0
EmailMissing            0
FlowerType_Orchid       0
FlowerType_Rose         0
FlowerType_Sunflower    0
FlowerType_Tulip        0
dtype: int64